Моделирование и эксперименты

Baseline model
Здесь будет использована логистическая регрессия, без фич

In [11]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

train_df = pd.read_csv('data/processed/train.csv')
val_df = pd.read_csv('data/processed/val.csv')

X_train_full = train_df.drop('Depression', axis=1)
y_train = train_df['Depression']
X_val_full = val_df.drop('Depression', axis=1)
y_val = val_df['Depression']

original_cols = ['Gender', 'Age', 'City', 'Profession', 'Academic Pressure', 'Work Pressure',
                 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Sleep Duration',
                 'Dietary Habits', 'Degree', 'Have you ever had suicidal thoughts ?',
                 'Work/Study Hours', 'Financial Stress', 'Family History of Mental Illness']

X_train_base = X_train_full[original_cols]
X_val_base = X_val_full[original_cols]

cat_features = X_train_base.select_dtypes(include=['object', 'category']).columns.tolist()
num_features = X_train_base.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ])

baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

baseline_model.fit(X_train_base, y_train)
y_pred = baseline_model.predict(X_val_base)

print("Baseline: Logistic Regression Results")
print(classification_report(y_val, y_pred))

Baseline: Logistic Regression Results
              precision    recall  f1-score   support

           0       0.83      0.80      0.82      1735
           1       0.86      0.88      0.87      2451

    accuracy                           0.85      4186
   macro avg       0.85      0.84      0.84      4186
weighted avg       0.85      0.85      0.85      4186



Сравнение моделей и ансамбли

In [ ]:
models = {
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
    'GradientBoosting': GradientBoostingClassifier(random_state=42)
}

named_estimators = []
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_train_full, y_train)
    preds = pipe.predict(X_val_full)
    print(f"{name} F1-Score: {f1_score(y_val, preds):.4f}")
    named_estimators.append((name, pipe))

ensemble = VotingClassifier(estimators=named_estimators, voting='soft')
ensemble.fit(X_train_full, y_train)
final_preds = ensemble.predict(X_val_full)

print(f"\nEnsemble (Voting) F1-Score: {f1_score(y_val, final_preds):.4f}")
print("\nОтчет по ансамблю:")
print(classification_report(y_val, final_preds))

RandomForest F1-Score: 0.8692
XGBoost F1-Score: 0.8678


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM F1-Score: 0.8737
GradientBoosting F1-Score: 0.8772
